# Task 0 — Data Pipeline

Run this notebook once to fully populate the database from scratch.  
Every step is **idempotent** — re-running skips what's already done.

| Step | What it does | Guard |
|---|---|---|
| **1. Tracks ingest** | `spotify_processed.parquet` → `tracks` hypertable | skips if rows already present |
| **2. News backfill** | Alpha Vantage weekly windows → `news_events` | skips weeks already in DB |
| **3. Lock sources** | Install DB triggers, revoke app-user writes | skips if triggers already exist |
| **4. Verify** | Row counts, date ranges, coverage summary | always runs |

> **Prerequisites**: `just up` and a populated `.env` (copy from `.env.example`).

In [ ]:
import time
import requests
import pandas as pd
from IPython.display import clear_output
from db import engine

API = 'http://localhost:8000'

def _bar(done, total, width=30):
    pct  = done / total if total else 0
    fill = int(pct * width)
    return f"[{'█' * fill}{'░' * (width - fill)}] {done}/{total} ({pct:.0%})"

try:
    h = requests.get(f'{API}/health', timeout=5).json()
    ok = h.get('db') == 'connected'
    print(f"API {h['status']}  ·  DB {'connected' if ok else 'UNAVAILABLE'}")
    if not ok:
        print('Run `just up` before continuing.')
except Exception as e:
    print(f'API unreachable: {e}\nRun `just up` first.')

---
## 1. Tracks ingest

Loads `spotify_processed.parquet` into the `tracks` hypertable.  
**Guard**: if the table already has rows, the ingest is skipped entirely — the parquet is a read-only source and must not be re-imported on top of existing data.

In [ ]:
existing = pd.read_sql('SELECT count(*) AS n FROM tracks', engine).iloc[0]['n']

if existing > 0:
    print(f'Tracks already loaded ({existing:,} rows) — skipping ingest.')
else:
    print('No tracks in DB — starting ingest ...')
    r = requests.post(f'{API}/ingest/tracks', timeout=10)
    r.raise_for_status()
    print(r.json()['message'])

    while True:
        s = requests.get(f'{API}/ingest/tracks/status', timeout=5).json()
        clear_output(wait=True)
        print(f'  running : {s["running"]}')
        if not s['running']:
            count = pd.read_sql('SELECT count(*) AS n FROM tracks', engine).iloc[0]['n']
            print(f'  done    : {count:,} rows in tracks')
            break
        time.sleep(8)

---
## 2. News backfill

Fetches weekly news from Alpha Vantage across 3 topics (entertainment, sports, economy).  
**Guard**: weeks already covered in `news_events` are skipped — the backfill is incremental by design.

> Free-tier rate: ~1 request / 13 s → ~57 min for the full 87-week window.  
> You can interrupt and resume — progress is preserved.

In [ ]:
DATE_FROM = '2023-10-18'
DATE_TO   = '2025-06-07'

cov = pd.read_sql('SELECT count(*) AS n FROM news_events', engine).iloc[0]['n']
s   = requests.get(f'{API}/ingest/news/status', timeout=5).json()

already_running = s.get('running', False)

# check if the full date range is already covered
db_cov = s.get('db_coverage') or {}
full_coverage = (
    db_cov.get('earliest') is not None
    and db_cov.get('earliest', '9999') <= DATE_FROM
    and db_cov.get('latest',   '0000') >= DATE_TO
)

if full_coverage and not already_running:
    print(f'News already fully covered ({cov:,} articles, {DATE_FROM} → {DATE_TO}) — skipping.')
elif not already_running:
    print(f'Starting news backfill {DATE_FROM} → {DATE_TO} ...')
    print(f'(existing: {cov:,} articles — incremental, skips covered weeks)')
    r = requests.post(f'{API}/ingest/news',
                      params={'date_from': DATE_FROM, 'date_to': DATE_TO}, timeout=10)
    r.raise_for_status()
    print(r.json()['message'])
else:
    print('Backfill already running — attaching to existing job ...')

In [ ]:
# live progress — re-run this cell at any time to refresh
# (or just leave it running; it polls until done)

while True:
    s   = requests.get(f'{API}/ingest/news/status', timeout=5).json()
    cov = s.get('db_coverage') or {}
    clear_output(wait=True)

    print(f"  running          : {s['running']}")
    if s['running']:
        done, total = s['weeks_done'], s['weeks_total']
        print(f"  progress         : {_bar(done, total)}")
        print(f"  current week     : {s.get('week_current') or '—'}")
        skipped = s.get('weeks_skipped', 0)
        if skipped:
            print(f"  weeks skipped    : {skipped} (already in DB)")
    print(f"  inserted this run: {s['articles_inserted']:,}")
    if cov.get('total_articles'):
        print(f"  total in DB      : {cov['total_articles']:,}")
        print(f"  date range       : {cov['earliest']} → {cov['latest']}")
    if s.get('last_error'):
        print(f"  error            : {s['last_error']}")

    if not s['running']:
        print('\nDone.' if s['articles_inserted'] >= 0 else '\nStopped.')
        break
    time.sleep(15)  # generous — free tier is slow anyway

---
## 3. Lock sources

Installs DB triggers that block `UPDATE`/`DELETE` on `tracks` and `news_events` for all users,
and revokes write access from the app user on those tables.

**Guard**: checks whether the triggers already exist — skips silently if they do.  
Run once, after both ingest and backfill are complete.

In [ ]:
trigger_exists = pd.read_sql("""
    SELECT count(*) AS n FROM pg_trigger
    WHERE  tgname = 'trg_prevent_tracks_modification'
""", engine).iloc[0]['n'] > 0

if trigger_exists:
    print('Source tables already locked — triggers in place, skipping.')
else:
    # make sure nothing is still ingesting
    t_running = requests.get(f'{API}/ingest/tracks/status', timeout=5).json()['running']
    n_running = requests.get(f'{API}/ingest/news/status',   timeout=5).json()['running']
    if t_running or n_running:
        print('An ingest is still running — wait for it to finish before locking.')
    else:
        r = requests.post(f'{API}/ingest/lock-sources', timeout=15)
        r.raise_for_status()
        print(r.json()['message'])
        print()
        print('Triggers installed:')
        triggers = pd.read_sql("""
            SELECT tgname, relname AS table_name
            FROM   pg_trigger t
            JOIN   pg_class c ON c.oid = t.tgrelid
            WHERE  tgname LIKE 'trg_prevent_%'
        """, engine)
        print(triggers.to_string(index=False))

---
## 4. Verify

In [ ]:
stats = pd.read_sql("""
    SELECT 'tracks'        AS "table", count(*)::text AS rows FROM tracks
    UNION ALL
    SELECT 'news_events',               count(*)::text         FROM news_events
    UNION ALL
    SELECT 'song_clusters',             count(*)::text         FROM song_clusters
    UNION ALL
    SELECT 'insights',                  count(*)::text         FROM insights
""", engine)

tracks_range = pd.read_sql("""
    SELECT min(snapshot_date)::text AS first_day,
           max(snapshot_date)::text AS last_day,
           count(DISTINCT country)  AS countries
    FROM   tracks
""", engine)

news_range = pd.read_sql("""
    SELECT min(published_at)::date::text AS first_article,
           max(published_at)::date::text AS last_article,
           count(DISTINCT published_at::date) AS days_covered
    FROM   news_events
""", engine)

locked = pd.read_sql("""
    SELECT count(*) AS n FROM pg_trigger WHERE tgname LIKE 'trg_prevent_%'
""", engine).iloc[0]['n']

print('── Row counts ───────────────────────────')
print(stats.to_string(index=False))
print()
print('── Tracks coverage ──────────────────────')
print(tracks_range.to_string(index=False))
print()
print('── News coverage ────────────────────────')
print(news_range.to_string(index=False))
print()
print(f'── Source protection ────────────────────')
print(f'   Triggers installed: {locked}  ({"locked" if locked >= 2 else "NOT locked — run Step 3"})')